# DD2360 Assignment 2
Runtime → Change runtime type → **T4 GPU** before running. All build output goes to `build/`.

In [2]:
%cd /content
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPO = "tattack99/DD2360-assignment-2"
REPO_DIR = "/content/repo"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        from google.colab import userdata
        token = userdata.get("GH-token")
        !git clone -q https://{token}@github.com/{REPO}.git {REPO_DIR}
    %cd {REPO_DIR}
    !git pull -q

/content
/content/repo


In [3]:
# Check that a GPU is attached and nvcc is available
!nvidia-smi
!nvcc --version

Thu Nov 19 06:35:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Q1 - 1D Convolution

In [ ]:
# Quick check: both kernels should print PASS
!make run

nvcc -arch=native -DTILE_WIDTH=256 -o build/hw2-convolution_tile256 assignment/hw2-convolution.cu
./build/hw2-convolution_tile256 1024
Timing - Allocated host memory                                          6 us
Timing - Allocated device memory                                   285048 us
Timing - Initialize arrays with values                                 37 us
Timing - Copying data to the GPU                                       77 us
Timing - Finished 1D convolution(basic)                               138 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         39 us
Timing - Finished 1D convolution(tiled)                                34 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         31 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CP

In [ ]:
# Q1.3 tile size: run every tile size on the largest input, compare the tiled kernel times
!make tilesweep ARGS=16384

=== TILE = 32 ===
nvcc -arch=native -DTILE_WIDTH=32 -o build/hw2-convolution_tile32 assignment/hw2-convolution.cu
./build/hw2-convolution_tile32 16384
Timing - Allocated host memory                                          7 us
Timing - Allocated device memory                                   281239 us
Timing - Initialize arrays with values                                497 us
Timing - Copying data to the GPU                                       92 us
Timing - Finished 1D convolution(basic)                               144 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         56 us
Timing - Finished 1D convolution(tiled)                                40 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results        121 us
Basic kernel vs CPU: PASS (0 mismatches)
Ti

In [ ]:
# Q1.3 screenshot: N = 1024 ... 16384 with the chosen tile size
TILE = 256
!make sweep TILE={TILE}

=== N = 1024, TILE = 256 ===
Timing - Allocated host memory                                          7 us
Timing - Allocated device memory                                   285778 us
Timing - Initialize arrays with values                                 31 us
Timing - Copying data to the GPU                                       41 us
Timing - Finished 1D convolution(basic)                               141 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         38 us
Timing - Finished 1D convolution(tiled)                                36 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         33 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CPU: PASS (0 mismatches)
Timing - Free memory resources                                        128 us
=== N 

In [ ]:
# Q1.4 Nsight Systems: where the time goes
!make profile TILE={TILE} ARGS=16384

nsys profile --stats=true --force-overwrite true -o build/hw2-convolution-report ./build/hw2-convolution_tile256 16384
Timing - Allocated host memory                                         11 us
Timing - Allocated device memory                                   534250 us
Timing - Initialize arrays with values                                416 us
Timing - Copying data to the GPU                                       98 us
Timing - Finished 1D convolution(basic)                               188 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         56 us
Timing - Finished 1D convolution(tiled)                                50 us
P_tiled[0] = 1.233154
P_tiled[1] = 1.405636
P_tiled[2] = 1.397376
P_tiled[3] = 1.596605
P_tiled[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         78 us
Basic kernel vs CPU: PASS (0 mismatches)
Tiled kernel vs CPU: PASS (0 misma

In [ ]:
# Q1.4 Nsight Compute: occupancy, shared memory, memory throughput
!make ncu TILE={TILE} ARGS=16384

ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section Occupancy --section LaunchStats ./build/hw2-convolution_tile256 16384
Timing - Allocated host memory                                          6 us
==PROF== Connected to process 12319 (/content/repo/build/hw2-convolution_tile256)
Timing - Allocated device memory                                   556311 us
Timing - Initialize arrays with values                                577 us
Timing - Copying data to the GPU                                       74 us
==PROF== Profiling "convolution_1D_basic" - 0: 0%....50%....100% - 9 passes
Timing - Finished 1D convolution(basic)                           2930668 us
P[0] = 1.233154
P[1] = 1.405636
P[2] = 1.397376
P[3] = 1.596605
P[4] = 1.297674
Timing - Copying output P to the CPU and print out the results         86 us
==PROF== Profiling "convolution_1D_tiled" - 1: 0%....50%....100% - 9 passes
Timing - Finished 1D convolution(tiled)                           1852466 us
P_tiled

## Question 2

In [7]:
%cd /content/repo/assignment
!ls

/content/repo/assignment
hw2-convolution.cu  matmul.cu


In [9]:
!nvcc -O2 -arch=native -o matmul matmul.cu

In [10]:
# Q2.2
!./matmul 1024 1024 1024 float 1 8 8 16 16 32 32

data type: float
matrix A: (1024 x 1024)
matrix B: (1024 x 1024)
matrix C: (1024 x 1024)
floating-point operations: 2*M*N*P = 2147483648
basic gemm: 16 x 16 threads per block, 4096 blocks, 1048576 threads launched, 0 exit idle
basic gemm global memory reads: 2*M*N*P = 2147483648
bytes copied host to device: 8388608 (A and B)
bytes copied device to host: 4194304 (C)
Achieved Occupancy / shared memory (one tile per run): make occupancy PROG=matmul ARGS="1024 1024 1024 float 0 8 8"
copy data from host to device: 2.628 ms
CPU reference: computing all 1048576 elements
CPU version: 3778.793 ms

kernel                      time (ms)    check   shared/block         global reads  details
gemm (16x16)                    8.966     PASS              0           2147483648  max abs err 9.15527e-05
tiled_gemm (8x8)               11.987     PASS          512 B            268435456  max abs err 9.15527e-05
tiled_gemm (16x16)              8.959     PASS         2048 B            134217728  max abs err 

In [11]:
# Q2.3
!./matmul 513 8192 1023 float 1 8 8 16 16 32 32

data type: float
matrix A: (513 x 8192)
matrix B: (8192 x 1023)
matrix C: (513 x 1023)
floating-point operations: 2*M*N*P = 8598306816
basic gemm: 16 x 16 threads per block, 2112 blocks, 540672 threads launched, 15873 exit idle
basic gemm global memory reads: 2*M*N*P = 8598306816
bytes copied host to device: 50331648 (A and B)
bytes copied device to host: 2099196 (C)
Achieved Occupancy / shared memory (one tile per run): make occupancy PROG=matmul ARGS="513 8192 1023 float 0 8 8"
copy data from host to device: 11.179 ms
CPU reference: checking 3 rows (3069 elements); a full CPU pass is slower than the GPU here
CPU version: 231.223 ms for 3 rows -> 39539.158 ms estimated for all 513 rows

kernel                      time (ms)    check   shared/block         global reads  details
gemm (16x16)                   29.989     PASS              0           8598306816  max abs err 0.000488281
tiled_gemm (8x8)               33.087     PASS          512 B           1074788352  max abs err 0.00048

In [12]:
# Q2.4 Nsight Systems: 1024^3 and 513x8192x1023 (CPU check off so it is quick)
!nsys profile --stats=true --force-overwrite true -o /tmp/mm_1024 ./matmul 1024 1024 1024 float 0 16 16
!nsys profile --stats=true --force-overwrite true -o /tmp/mm_513  ./matmul 513 8192 1023 float 0 16 16

data type: float
matrix A: (1024 x 1024)
matrix B: (1024 x 1024)
matrix C: (1024 x 1024)
floating-point operations: 2*M*N*P = 2147483648
basic gemm: 16 x 16 threads per block, 4096 blocks, 1048576 threads launched, 0 exit idle
basic gemm global memory reads: 2*M*N*P = 2147483648
bytes copied host to device: 8388608 (A and B)
bytes copied device to host: 4194304 (C)
Achieved Occupancy / shared memory (one tile per run): make occupancy PROG=matmul ARGS="1024 1024 1024 float 0 16 16"
copy data from host to device: 2.054 ms
CPU reference: skipped

kernel                      time (ms)    check   shared/block         global reads  details
gemm (16x16)                    8.967      n/a              0           2147483648  max abs err 0
tiled_gemm (16x16)              7.905      n/a         2048 B            134217728  max abs err 0

CSV,float,1024,1024,1024,0.000,0,8.9666,16,16,7.9054
Generating '/tmp/nsys-report-4603.qdstrm'
[1/8] [========================100%] mm_1024.nsys-rep
[2/8] [=====

In [13]:
# Q2.5
import subprocess

sizes = [64, 128, 256, 384, 512, 768, 1024, 1536, 2048, 3072]
with open("results.csv", "w") as f:
    for s in sizes:
        out = subprocess.run(
            ["./matmul", str(s), str(s), str(s), "float", "1", "8", "8", "16", "16", "32", "32"],
            capture_output=True, text=True).stdout
        print(out)
        for line in out.splitlines():
            if line.startswith("CSV,"):
                f.write(line + "\n")

data type: float
matrix A: (64 x 64)
matrix B: (64 x 64)
matrix C: (64 x 64)
floating-point operations: 2*M*N*P = 524288
basic gemm: 16 x 16 threads per block, 16 blocks, 4096 threads launched, 0 exit idle
basic gemm global memory reads: 2*M*N*P = 524288
bytes copied host to device: 32768 (A and B)
bytes copied device to host: 16384 (C)
Achieved Occupancy / shared memory (one tile per run): make occupancy PROG=matmul ARGS="64 64 64 float 0 8 8"
copy data from host to device: 0.043 ms
CPU reference: computing all 4096 elements
CPU version: 0.251 ms

kernel                      time (ms)    check   shared/block         global reads  details
gemm (16x16)                    0.007     PASS              0               524288  max abs err 3.8147e-06
tiled_gemm (8x8)                0.018     PASS          512 B                65536  max abs err 3.8147e-06
tiled_gemm (16x16)              0.014     PASS         2048 B                32768  max abs err 3.8147e-06
tiled_gemm (32x32)              

In [14]:
!python3 plot_results.py results.csv
from IPython.display import Image
Image("runtime_comparison.png")

python3: can't open file '/content/repo/assignment/plot_results.py': [Errno 2] No such file or directory


FileNotFoundError: No such file or directory: 'runtime_comparison.png'

FileNotFoundError: No such file or directory: 'runtime_comparison.png'

<IPython.core.display.Image object>

# Question 3

In [ ]:
%cd /content/repo/llama2.c/


/content/repo/llama2.c


In [ ]:
#We made a modified run.c to include a counter of matmul calls.
!gcc -Ofast -o run my_run.c -lm

In [ ]:
model = "stories15M" #We want the small model only
download_url = "https://huggingface.co/karpathy/tinyllamas/resolve/main/stories15M.bin"
!wget $download_url
model_file = model + ".bin"

--2026-11-18 21:10:07--  https://huggingface.co/karpathy/tinyllamas/resolve/main/stories15M.bin
Resolving huggingface.co (huggingface.co)... 3.170.185.33, 3.170.185.35, 3.170.185.14, ...
Connecting to huggingface.co (huggingface.co)|3.170.185.33|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/64c19eb0275dc8cd7f170b8f/07dbdd97d4def5722a8deaeb97073a464e041687313cbf6b06cccb9efdd0c4b7?xip=EtyJAokU1l4&user_id=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27stories15M.bin%3B+filename%3D%22stories15M.bin%22%3B&response-content-type=application%2Foctet-stream&X-Xet-Cas-Uid=public&Expires=1795039807&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjRjMTllYjAyNzVkYzhjZDdmMTcwYjhmLzA3ZGJkZDk3ZDRkZWY1NzIyYThkZWFlYjk3MDczYTQ2NGUwNDE2ODczMTNjYmY2YjA2Y2NjYjllZmRkMGM0YjdcXD94aXA9RXR5SkFva1UxbDQmdXNlcl9pZD1wdWJsaWMmcmVzcG9uc2UtY29udGVudC1kaXNwb3NpdGlvbj0qJnJlc3BvbnNl

In [ ]:
#@title Generate Stories

# Generate args
max_token = 256 #@param {type:"slider", min:32, max:1024, step:32}
temperature = 0.8 #@param {type:"slider", min:0.0, max:1, step:0.05}
top_p = 0.9 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
prompt = "One day, Lily met a Shoggoth" #@param {type:"string"}

print(f"model: {model_file}, max_token: {max_token}, temperature: {temperature}, top_p: {top_p}, prompt: {prompt}")
print(f"----------------------------\n")

cmd = f'./run {model_file} -t {temperature} -p {top_p} -n {max_token} -i "{prompt}"'
!{cmd}

model: stories15M.bin, max_token: 256, temperature: 0.8, top_p: 0.9, prompt: One day, Lily met a Shoggoth
----------------------------

One day, Lily met a Shoggoth in the park. She was very excited and asked her mom if she could keep it. Her mom said yes and showed her a shiny red penny. 
Lily was very happy and hugged her mom. She put the penny in her pocket and ran off to play.
When Lily got to the park, she saw a new boy. He was playing with his friends and having lots of fun. Lily wanted to play with him, but the boy didn't want to play with her. He looked at her pocket and said, "That's mine! You can't have it!"
Lily was very sad and started to cry. Her mom saw her crying and asked her what happened. Lily told her about the boy and his friends. Her mom said, "It's not nice to be envious of someone else's things. You should try to find someone who would appreciate it and share it with you."
Lily thought about it and decided to give the penny to the boy. He was so happy and thanked

#Porting to GPU


In [ ]:
!make runcuda_1

gcc -O3 -c runcuda.c -o runcuda.o
nvcc -O3 -c matmul.cu -o matmul.o
make: nvcc: No such file or directory
make: *** [Makefile:9: runcuda_1] Error 127


In [ ]:
!./runcuda_1 -n 256 -i "Once in a dark forest"

Usage:   run <checkpoint> [options]
Example: run model.bin -n 256 -i "Once upon a time"
Options:
  -t <float>  temperature in [0,inf], default 1.0
  -p <float>  p value in top-p (nucleus) sampling in [0,1] default 0.9
  -s <int>    random seed, default time(NULL)
  -n <int>    number of steps to run for, default 256. 0 = max_seq_len
  -i <string> input prompt
  -z <string> optional path to custom tokenizer
  -m <string> mode: generate|chat, default: generate
  -y <string> (optional) system prompt in chat mode


In [ ]:
#@title Run Meta's Llama 2 models - Not sure we do this...

#@markdown input your huggingface [access token](https://huggingface.co/settings/tokens) to download Meta's Llama 2 models.

from huggingface_hub import snapshot_download

token = "replace your huggingface access token" #@param {type:"string"}
path = snapshot_download(repo_id="meta-llama/Llama-2-7b",cache_dir="Llama-2-7b", use_auth_token=token)

!python export.py llama2_7b.bin --meta-llama $path

print("./run llama2_7b.bin\n")
!./run llama2_7b.bin

TypeError: snapshot_download() got an unexpected keyword argument 'use_auth_token'